## Обзор данных

In [2]:
import os
from pathlib import Path

import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display

rng = np.random.default_rng(2026)
os.environ['KAGGLEHUB_CACHE'] = str(Path.cwd())
dataset_path = kagglehub.dataset_download('ammarhezamahmed/malaysian-telecommunication-google-play-reviews')
print('Датасет скачан в папку:', dataset_path)
print('Файлы в датасете:', os.listdir(dataset_path))

Датасет скачан в папку: c:\Users\Zenbook\Desktop\3rd course\Reviews-processing\datasets\ammarhezamahmed\malaysian-telecommunication-google-play-reviews\versions\1
Файлы в датасете: ['Malaysian_Telecommunication_Google Play_reviews.csv']


In [3]:
csv_path = Path(dataset_path) / 'Malaysian_Telecommunication_Google Play_reviews.csv'
data = pd.read_csv(csv_path)

display(data.head())
data.info()

,userName,content,score,thumbsUpCount,at,provider
0,Cheo Yan Ho,"Terrible app, when I want to pay the bill usin...",1,181,2023-07-02,maxis
1,Syed Husayn,"sooo unstable, buggy and slow (not surprising ...",1,116,2023-10-03,maxis
2,Muhammad Amirul,"The app took so very long to sync, it's better...",1,429,2022-11-12,maxis
3,Hakimi Kimi,"This app is now officially became one of the ""...",1,261,2023-01-13,maxis
4,Catherina Erni,This apps is good but I don't like it when it ...,2,143,2020-06-07,maxis


<class 'pandas.DataFrame'>
RangeIndex: 159444 entries, 0 to 159443
Data columns (total 6 columns):
 #   Column         Non-Null Count   Dtype
---  ------         --------------   -----
 0   userName       159443 non-null  str  
 1   content        159428 non-null  str  
 2   score          159444 non-null  int64
 3   thumbsUpCount  159444 non-null  int64
 4   at             159444 non-null  str  
 5   provider       159444 non-null  str  
dtypes: int64(2), str(4)
memory usage: 20.4 MB


In [4]:
display(data['score'].value_counts(dropna=False).sort_index())

score
0        1
1    64659
2     9164
3     9422
4    11019
5    65179
Name: count, dtype: int64

In [5]:
df = data[['content', 'score']].copy()

df = df.dropna(subset=['content', 'score'])
df['content'] = df['content'].str.strip()
df = df[df['content'].ne('') & df['score'].isin([1, 2, 3, 4, 5])].copy()

df['sentiment'] = df['score'].map({
    1: 'negative',
    2: 'negative',
    3: 'neutral',
    4: 'positive',
    5: 'positive'
})

print('Осталось отзывов:', len(df))
display(df['sentiment'].value_counts())

df['text_key'] = df['content'].str.casefold().str.replace(
    r'\s+', ' ', regex=True
)
print('Повторов текста:', df['text_key'].duplicated().sum())
print('Текстов с разными метками:',
      df.groupby('text_key')['sentiment'].nunique().gt(1).sum())

Осталось отзывов: 159427


sentiment
positive    76185
negative    73822
neutral      9420
Name: count, dtype: int64

Повторов текста: 56434
Текстов с разными метками: 1120


In [6]:
# Находим тексты, которым соответствуют разные классы
label_counts = df.groupby('text_key')['sentiment'].nunique()
conflict_keys = label_counts[label_counts > 1].index

# Берём 5 таких текстов и показываем их оценки и метки
examples = (
    df[df['text_key'].isin(conflict_keys[:5])]
    [['content', 'score', 'sentiment', 'text_key']]
    .drop_duplicates(subset=['text_key', 'score'])
    .sort_values(['text_key', 'score'])
)

with pd.option_context('display.max_colwidth', None):
    display(examples[['content', 'score', 'sentiment']])

,content,score,sentiment
241,#NAME?,1,negative
55110,#NAME?,4,positive
15896,0k,3,neutral
21918,0k,4,positive
13669,0k,5,positive
65138,4 star,3,neutral
59784,4 star,4,positive
53717,a bit slow,3,neutral
54564,a bit slow,5,positive
150166,Aaa,3,neutral


In [7]:
# Убираем технический маркер
df_valid = df[df['text_key'] != '#name?'].copy()

# Находим конфликтующие метки
label_counts = df_valid.groupby('text_key')['sentiment'].nunique()
conflict_keys = label_counts[label_counts > 1].index

# Исключаем все строки с конфликтующими текстами
df_consistent = df_valid[
    ~df_valid['text_key'].isin(conflict_keys)
].copy()

# Оставляем один экземпляр каждого текста
df_clean = df_consistent.drop_duplicates('text_key').copy()

print('Удалено строк #NAME?:', len(df) - len(df_valid))
print('Исключено строк с конфликтами:',
      len(df_valid) - len(df_consistent))
print('Удалено оставшихся повторов:',
      len(df_consistent) - len(df_clean))
print('Осталось для работы:', len(df_clean))

display(df_clean['sentiment'].value_counts())

Удалено строк #NAME?: 4
Исключено строк с конфликтами: 49637
Удалено оставшихся повторов: 7913
Осталось для работы: 101873


sentiment
negative    65779
positive    29408
neutral      6686
Name: count, dtype: int64

In [8]:
examples = (
    df_clean.groupby('sentiment')
    .sample(n=5, random_state=42)
    .sort_values('sentiment')
)

with pd.option_context('display.max_colwidth', None):
    display(examples[['content', 'score', 'sentiment']])

,content,score,sentiment
62310,Makin teruk line celcom sekarang kenapa ye....,1,negative
131560,Forcing customer to update app...very bad,1,negative
4326,Can't even login.,1,negative
1510,please dont give free 10GB facebook if still waste my all data.. im always open facebook everyday but my free 10gb facebook not decrease even 1kb.. then my data done because of facebook.. this is all liar.. please give a best service to customer and dont lie for business.. free 10gb facebook like no function at all.. wasting money.. hope can fix this problem.. thanks alots...,1,negative
14912,"Dah ade internet, tapi masih lembap slow, apehal ni",1,negative
14313,Good jop,3,neutral
26713,Tak suka reward yang sekarang.,3,neutral
70054,Bila buka internet rasa mcm nk baling phone ja..itulah yang saya rasa sekarang.n,3,neutral
84530,One of my Xpax number doesn't appear credit balance. It show blank credit \nbalance space.,3,neutral
45558,Couldve been better... Loading a bit slow... More interactive pls... N pls enable realtime chatting with ur customer care personnel,3,neutral


In [9]:
%pip install langdetect

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: C:\Users\Zenbook\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [10]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

assert df_clean['text_key'].is_unique

train_df, remaining_df = train_test_split(
    df_clean,
    test_size=0.30,
    random_state=42,
    stratify=df_clean['sentiment']
)

val_df, test_df = train_test_split(
    remaining_df,
    test_size=0.50,
    random_state=42,
    stratify=remaining_df['sentiment']
)

print('Train:', len(train_df))
print('Validation:', len(val_df))
print('Test:', len(test_df))

baseline = make_pipeline(
    TfidfVectorizer(max_features=30000, min_df=2),
    LogisticRegression(max_iter=1000, class_weight='balanced')
)

baseline.fit(train_df['content'], train_df['sentiment'])

val_pred = baseline.predict(val_df['content'])

print(classification_report(
    val_df['sentiment'],
    val_pred,
    digits=3,
    zero_division=0
))

Train: 71311
Validation: 15281
Test: 15281
              precision    recall  f1-score   support

    negative      0.908     0.727     0.807      9867
     neutral      0.134     0.395     0.200      1003
    positive      0.787     0.788     0.788      4411

    accuracy                          0.723     15281
   macro avg      0.610     0.637     0.598     15281
weighted avg      0.822     0.723     0.762     15281



In [11]:
from sklearn.base import clone

models = {
    'Baseline': baseline,
    'EXP-01: пары слов': clone(baseline).set_params(
        tfidfvectorizer__ngram_range=(1, 2)
    ),
    'EXP-02: без балансировки': clone(baseline).set_params(
        logisticregression__class_weight=None
    )
}

experiment_rows = []

def record_metrics(name, predictions):
    report = classification_report(
        val_df['sentiment'],
        predictions,
        output_dict=True,
        zero_division=0
    )
    experiment_rows.append({
        'Вариант': name,
        'Macro-F1': report['macro avg']['f1-score'],
        'Recall negative': report['negative']['recall'],
        'Precision negative': report['negative']['precision'],
        'F1 neutral': report['neutral']['f1-score']
    })

# Простое правило для сравнения: всегда самый частый класс train
majority_class = train_df['sentiment'].mode()[0]
record_metrics(
    f'Всегда {majority_class}',
    [majority_class] * len(val_df)
)

for name, model in models.items():
    print('Проверяем:', name)

    if name != 'Baseline':
        model.fit(train_df['content'], train_df['sentiment'])

    predictions = model.predict(val_df['content'])
    record_metrics(name, predictions)

experiment_log = pd.DataFrame(experiment_rows)
display(experiment_log.round(3))

Проверяем: Baseline
Проверяем: EXP-01: пары слов
Проверяем: EXP-02: без балансировки


,Вариант,Macro-F1,Recall negative,Precision negative,F1 neutral
0,Всегда negative,0.262,1.000,0.646,0.000
1,Baseline,0.598,0.727,0.908,0.200
2,EXP-01: пары слов,0.610,0.788,0.903,0.196
3,EXP-02: без балансировки,0.576,0.945,0.854,0.023


In [12]:
selected_model = models['EXP-02: без балансировки']

# Итоговая проверка выбранной модели
test_pred = selected_model.predict(test_df['content'])

print('Итоговые метрики на TEST:')
print(classification_report(
    test_df['sentiment'], test_pred,
    digits=3, zero_division=0
))

# Создаём CSV для демонстрации из 20 validation-отзывов
val_df[['content']].sample(
    n=20, random_state=42
).to_csv('demo_reviews.csv', index=False)

# Здесь можно указать путь к своему CSV со столбцом content
input_path = 'demo_reviews.csv'
reviews = pd.read_csv(input_path)

if 'content' not in reviews.columns:
    raise ValueError('В CSV нужен столбец content.')

if reviews.empty:
    raise ValueError('В CSV нет отзывов.')

texts = reviews['content'].astype('string').fillna('').str.strip()

if texts.eq('').any():
    raise ValueError('В CSV есть пустые отзывы — заполните или удалите их.')

# Предсказываем только по тексту
reviews['sentiment'] = selected_model.predict(texts)

# Фильтр негативных отзывов
negative_reviews = reviews[
    reviews['sentiment'] == 'negative'
].copy()

reviews.to_csv('all_predictions.csv', index=False, encoding='utf-8-sig')
negative_reviews.to_csv(
    'negative_reviews.csv', index=False, encoding='utf-8-sig'
)

print('Обработано:', len(reviews))
print('Негативных:', len(negative_reviews))
print('Сохранено: all_predictions.csv и negative_reviews.csv')

display(reviews)
display(negative_reviews)

Итоговые метрики на TEST:
              precision    recall  f1-score   support

    negative      0.855     0.948     0.899      9867
     neutral      0.311     0.014     0.027      1003
    positive      0.825     0.804     0.814      4411

    accuracy                          0.845     15281
   macro avg      0.664     0.589     0.580     15281
weighted avg      0.811     0.845     0.817     15281

Обработано: 20
Негативных: 14
Сохранено: all_predictions.csv и negative_reviews.csv


,content,sentiment
0,I'M USING ABOUT RM 170 THIS MONTH FOR GAMING T...,negative
1,if only this app provide payment via fpx or on...,positive
2,Your live chat really need improvements. Whene...,negative
3,"just uninstalled this. this eternally lagging,...",negative
4,after this new update my screen turn white.......,negative
5,Bagus :),positive
6,"kenapa tak boleh claim rewards? kalau tak, bol...",negative
7,Apps all is good. Please improve on tele line ...,negative
8,I can say maxis very good and strong caverigh,positive
9,not bad...tukar talian lagi bagus la...dh susa...,negative


,content,sentiment
0,I'M USING ABOUT RM 170 THIS MONTH FOR GAMING T...,negative
2,Your live chat really need improvements. Whene...,negative
3,"just uninstalled this. this eternally lagging,...",negative
4,after this new update my screen turn white.......,negative
6,"kenapa tak boleh claim rewards? kalau tak, bol...",negative
7,Apps all is good. Please improve on tele line ...,negative
9,not bad...tukar talian lagi bagus la...dh susa...,negative
10,blank..,negative
11,"Dah subscribe plan mahal pun, masih lembab",negative
13,Problem login..so many update..,negative


In [13]:
import joblib
from pathlib import Path

joblib.dump(selected_model, "sentiment_model.joblib")

print("Модель сохранена в:", Path("sentiment_model.joblib").resolve())

Модель сохранена в: C:\Users\Zenbook\Desktop\3rd course\Reviews-processing\sentiment_model.joblib


In [14]:
%pip install streamlit

Defaulting to user installation because normal site-packages is not writeable
  Using cached streamlit-1.66.1-py3-none-any.whl.metadata (10 kB)
  Using cached altair-6.3.0-py3-none-any.whl.metadata (11 kB)
  Using cached pydeck-0.9.3-py2.py3-none-any.whl.metadata (4.2 kB)
  Using cached starlette-1.10.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached python_multipart-0.0.32-py3-none-any.whl.metadata (2.1 kB)
  Using cached itsdangerous-2.2.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached watchdog-6.0.0-py3-none-win_amd64.whl.metadata (44 kB)
Using cached streamlit-1.66.1-py3-none-any.whl (10.4 MB)
Using cached altair-6.3.0-py3-none-any.whl (797 kB)
Using cached itsdangerous-2.2.0-py3-none-any.whl (16 kB)
Using cached pydeck-0.9.3-py2.py3-none-any.whl (11.4 MB)
Using cached python_multipart-0.0.32-py3-none-any.whl (30 kB)
Using cached starlette-1.10.0-py3-none-any.whl (80 kB)
Using cached watchdog-6.0.0-py3-none-win_amd64.whl (79 kB)

   ---------------------------------------- 0/7


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: C:\Users\Zenbook\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip
